## Setup: mount Google Drive

Data files are expected directly in `My Drive` root. Run this cell to mount Drive and verify.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for fname in ["transactions_data.csv", "users_data.csv", "cards_data.csv", "mcc_codes.json", "train_fraud_labels.json"]:
    assert os.path.isfile(f"/content/drive/MyDrive/{fname}"), f"Missing {fname} in My Drive root"
print("All data files found.")

In [1]:
import pandas as pd
import numpy as np

In [15]:
transactions = pd.read_csv("/content/drive/MyDrive/transactions_data.csv")

print("Transactions loaded:", transactions.shape)

Transactions loaded: (13305915, 12)


In [17]:
transactions = transactions[
    [
        "id",
        "date",
        "client_id",
        "card_id",
        "amount",
        "use_chip",
        "merchant_id",
        "mcc",
        "errors"
    ]
].copy()

In [13]:
transactions["date"] = pd.to_datetime(transactions["date"])

In [14]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 9 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   client_id    int64         
 3   card_id      int64         
 4   amount       float64       
 5   use_chip     str           
 6   merchant_id  int64         
 7   mcc          int64         
 8   errors       str           
dtypes: datetime64[us](1), float64(1), int64(5), str(2)
memory usage: 913.6 MB


In [11]:
print(transactions["amount"].dtype)
print(transactions["amount"].head())

float64
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: amount, dtype: float64


In [10]:
transactions["amount"] = (
    transactions["amount"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

AttributeError: Can only use .str accessor with string values, not floating

Verify the amount column cleaned correctly.

In [ ]:
print("Amount dtype:", transactions["amount"].dtype)
print(transactions["amount"].describe())
print("Missing amounts:", transactions["amount"].isna().sum())

## Encode categorical fields

Flag failed transactions, one-hot encode `use_chip`, and merge in MCC merchant category descriptions.

In [ ]:
transactions["has_error"] = transactions["errors"].notna().astype(int)

In [ ]:
transactions = pd.get_dummies(
    transactions,
    columns=["use_chip"],
    prefix="channel"
)

In [ ]:
import json as _json

with open("/content/drive/MyDrive/mcc_codes.json", "r", encoding="utf-8") as f:
    mcc_codes = _json.load(f)

mcc_lookup = pd.DataFrame(
    {"mcc": [int(k) for k in mcc_codes.keys()], "mcc_description": list(mcc_codes.values())}
)

transactions = transactions.merge(mcc_lookup, on="mcc", how="left")
print("Unmatched MCC codes:", transactions["mcc_description"].isna().sum())

## Derive time-based features

In [ ]:
transactions["hour"] = transactions["date"].dt.hour
transactions["day_of_week"] = transactions["date"].dt.dayofweek
transactions["month"] = transactions["date"].dt.month
transactions["is_weekend"] = transactions["day_of_week"].isin([5, 6]).astype(int)

## Per-client behavioural aggregation

The raw data has no worker/client role or job/dispute columns, so we aggregate each `client_id`'s real transaction behaviour into proxy features (transaction volume, error rate, and time between transactions), then use those as the basis for synthetic worker and client trust datasets below.

In [ ]:
transactions = transactions.sort_values(["client_id", "date"])
transactions["prev_date"] = transactions.groupby("client_id")["date"].shift(1)
transactions["hours_since_prev"] = (
    (transactions["date"] - transactions["prev_date"]).dt.total_seconds() / 3600
)

client_agg = transactions.groupby("client_id").agg(
    transaction_count=("id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
    error_count=("has_error", "sum"),
    avg_hours_between_transactions=("hours_since_prev", "mean"),
).reset_index()

client_agg["error_rate"] = client_agg["error_count"] / client_agg["transaction_count"]
client_agg["avg_hours_between_transactions"] = client_agg["avg_hours_between_transactions"].fillna(
    client_agg["avg_hours_between_transactions"].median()
)

print("Per-client aggregate rows:", client_agg.shape)
client_agg.describe()

## Synthetic worker / client trust datasets

This dataset has no worker/client role, and the proposal's trust features (job completion rate, dispute frequency, response time for workers; payment completion, escrow release time, refund/dispute history for clients) don't exist in real platform data yet. As the proposal anticipates, we generate synthetic behaviour: each `client_id` is randomly assigned a role, and role-appropriate proxy features are derived from that client's real transaction statistics above (error rate as a dispute/refund proxy, average time between transactions as a response/release-time proxy, transaction volume/amount as a completion proxy).

In [ ]:
rng = np.random.default_rng(seed=42)
client_agg["role"] = rng.choice(["worker", "client"], size=len(client_agg))
print(client_agg["role"].value_counts())

In [ ]:
worker_df = client_agg[client_agg["role"] == "worker"].copy()

worker_df["job_completion_rate"] = 1 - worker_df["error_rate"]
worker_df["dispute_count"] = worker_df["error_count"]
worker_df["response_time_hours"] = worker_df["avg_hours_between_transactions"]
worker_df["total_jobs"] = worker_df["transaction_count"]

worker_dataset = worker_df[
    ["client_id", "job_completion_rate", "dispute_count", "response_time_hours", "total_jobs"]
].reset_index(drop=True)

print("Worker dataset:", worker_dataset.shape)
worker_dataset.head()

In [ ]:
client_df = client_agg[client_agg["role"] == "client"].copy()

client_df["payment_completion_rate"] = 1 - client_df["error_rate"]
client_df["escrow_release_time_hours"] = client_df["avg_hours_between_transactions"]
client_df["refund_requests"] = client_df["error_count"]
client_df["total_jobs_paid"] = client_df["transaction_count"]

client_dataset = client_df[
    ["client_id", "payment_completion_rate", "escrow_release_time_hours", "refund_requests", "total_jobs_paid"]
].reset_index(drop=True)

print("Client dataset:", client_dataset.shape)
client_dataset.head()

## Export engineered datasets

Saved locally under `data/processed/` (gitignored) for `src/train.py` to load.

In [ ]:
from pathlib import Path

processed_dir = Path("/content/drive/MyDrive/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

worker_dataset.to_csv(processed_dir / "worker_features.csv", index=False)
client_dataset.to_csv(processed_dir / "client_features.csv", index=False)

print("Saved:", processed_dir / "worker_features.csv", worker_dataset.shape)
print("Saved:", processed_dir / "client_features.csv", client_dataset.shape)